In [8]:
y_true.

AttributeError: 'numpy.ndarray' object has no attribute 'unique'

In [7]:
y_scores

array([[1. , 0. , 0. ],
       [1. , 0. , 0. ],
       [0.1, 0.6, 0.3],
       [0.1, 0.6, 0.3],
       [0.1, 0.6, 0.3],
       [0.1, 0.6, 0.3]])

In [2]:
import numpy as np
from sklearn.metrics import precision_recall_curve, auc

# 假设你有真实标签和预测分数
y_true = np.array([0, 0, 1, 1,2,2])
y_scores = np.array([[1,0,0],[1,0,0], [0.1,0.6,0.3],[0.1,0.6,0.3],[0.1,0.6,0.3],[0.1,0.6,0.3] ])

# 计算 precision-recall 对
precision, recall, _ = precision_recall_curve(y_true, y_scores)

# 计算 AUC
pr_auc = auc(recall, precision)

print(f'Precision-Recall AUC: {pr_auc}')


ValueError: multiclass format is not supported

In [3]:
import numpy as np
import numpy as np
from sklearn.metrics import precision_recall_curve, auc

def compute_binary_pr_auc(reference, predict_logits):
    precision, recall, _ = precision_recall_curve(reference, predict_logits)
    return auc(recall, precision)

def compute_ovr_pr_auc(reference, predict_logits, average=None,ignore_idx=[]):
    n_classes = predict_logits.shape[1]
    pr_aucs = []
    for class_idx in range(n_classes):
        if class_idx not in ignore_idx:
            pr_auc = compute_binary_pr_auc((reference == class_idx).astype(int), predict_logits[:, class_idx])
            pr_aucs.append(pr_auc)
    if average == "macro":
        return np.mean(pr_aucs)
    elif average == "weighted":
        class_counts = np.bincount(reference)
        weighted_pr_aucs = np.array(pr_aucs) * class_counts / len(reference)
        return np.sum(weighted_pr_aucs)
    else:
        return pr_aucs

def compute_ovo_pr_auc(reference, predict_logits, average=None):
    # OvO is not directly supported by precision_recall_curve
    raise NotImplementedError("OvO PR AUC computation is not implemented yet.")

def pr_auc_score(reference, predict_logits, multi_class=None, average=None):
    if multi_class == "ovr":
        pr_auc = compute_ovr_pr_auc(reference, predict_logits, average=average)
    elif multi_class == "ovo":
        pr_auc = compute_ovo_pr_auc(reference, predict_logits, average=average)
    else:
        pr_auc = compute_binary_pr_auc(reference, predict_logits)
    return pr_auc

In [9]:
pr_auc_score(y_true,y_scores,"ovr")

[1.0, 0.75, 0.75]

In [1]:
from networkx import trophic_differences
import torch
import torch.nn as nn
from typing import Optional, Tuple

def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat((-x2, x1), dim=-1)


def apply_rotary_pos_emb(x, cos, sin):
    """
    The function applies rotary positional embedding to the input tensor using cosine and sine values.

    :param x: The input tensor x. It is a 4-dimensional tensor with shape (batch_size, num_channels,
    height, width)
    :param cos: A tensor representing the cosine values for the rotary positional embedding. It has
    shape (batch_size, num_heads, sequence_length, embedding_dim)
    :param sin: The `sin` parameter is a tensor representing the sine values for the rotary positional
    embedding. It has shape `(batch_size, num_heads, sequence_length, embedding_dim)`
    :return: the result of applying the rotary positional embedding to the input tensor x.
    """
    cos = cos[:, :, : x.shape[-2], :]
    sin = sin[:, :, : x.shape[-2], :]
    return (x * cos) + (rotate_half(x) * sin)


class RotaryEmbedding(torch.nn.Module):
    """
    Rotary position embeddings based on those in
    [RoFormer](https://huggingface.co/docs/transformers/model_doc/roformer). Query and keys are transformed by rotation
    matrices which depend on their relative positions.
    """

    def __init__(self, dim: int):
        super().__init__()
        # Generate and save the inverse frequency buffer (non trainable)
        inv_freq = 1.0 / (10000 ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer("inv_freq", inv_freq)

        self._seq_len_cached = None
        self._cos_cached = None
        self._sin_cached = None

    def _update_cos_sin_tables(self, x, seq_dimension=2):
        seq_len = x.shape[seq_dimension]

        # Reset the tables if the sequence length has changed,
        # or if we're on a new device (possibly due to tracing for instance)
        if seq_len != self._seq_len_cached or self._cos_cached.device != x.device:
            self._seq_len_cached = seq_len
            t = torch.arange(x.shape[seq_dimension], device=x.device).type_as(
                self.inv_freq
            )
            freqs = torch.outer(t, self.inv_freq)
            emb = torch.cat((freqs, freqs), dim=-1).to(x.device)

            self._cos_cached = emb.cos()[None, None, :, :]
            self._sin_cached = emb.sin()[None, None, :, :]

        return self._cos_cached, self._sin_cached

    def forward(
        self, q: torch.Tensor, k: torch.Tensor
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        self._cos_cached, self._sin_cached = self._update_cos_sin_tables(
            k, seq_dimension=-2
        )

        return (
            apply_rotary_pos_emb(q, self._cos_cached, self._sin_cached),
            apply_rotary_pos_emb(k, self._cos_cached, self._sin_cached),
        )


def transpose_for_scores(x,num_attention_heads,attention_head_size) -> torch.Tensor:
    new_x_shape = x.size()[:-1] + (
        num_attention_heads,
        attention_head_size,
    )
    x = x.view(new_x_shape)
    return x.permute(0, 2, 3, 1)#[B,S,Hi]->[B,He,S,Hid]

class SelfAttention(nn.Module):
    def __init__(self,config):
        self.num_attention_heads = config.num_attention_heads
        self.attention_head_size = config.hidden_size // num_attention_heads
        self.all_head_size = self.num_attention_heads * attention_head_size
            
        self.query1   = nn.Linear(hidden_size, all_head_size, bias=False)
        self.key1     = nn.Linear(hidden_size, all_head_size, bias=False)

        self.query2   = nn.Linear(hidden_size, all_head_size, bias=False)
        self.key2     = nn.Linear(hidden_size, all_head_size, bias=False)
        self.value   = nn.Linear(hidden_size, all_head_size, bias=False)
        
        self.rotary_embeddings = RotaryEmbedding(dim=attention_head_size)
        self.filter=nn.Parameter(torch.tensor([1/self.attention_head_size]))
        self.lambda_k=nn.Parameter(torch.tensor([1/self.attention_head_size]))
    def forward(self,hidden_states,attention_mask=None,head_mask=None,weight_mask=None):
        value_layer  = transpose_for_scores(self.value(hidden_states) ,self.num_attention_heads,self.attention_head_size)
        query_layer1 = transpose_for_scores(self.query1(hidden_states),self.num_attention_heads,self.attention_head_size)
        key_layer1   = transpose_for_scores(self.key1(hidden_states)  ,self.num_attention_heads,self.attention_head_size)
        query_layer2 = transpose_for_scores(self.query2(hidden_states),self.num_attention_heads,self.attention_head_size)
        key_layer2   = transpose_for_scores(self.key2(hidden_states)  ,self.num_attention_heads,self.attention_head_size)
        if weight_mask is not None:
            query_layer1 =torch.matmul(query_layer1,    weight_mask).transpose(-1, -2)
            key_layer1   =torch.matmul(key_layer1,      weight_mask).transpose(-1, -2)
            query_layer2 =torch.matmul(query_layer2,    weight_mask).transpose(-1, -2)
            key_layer2   =torch.matmul(key_layer2,      weight_mask).transpose(-1, -2)
            value_layer  =torch.matmul(value_layer,     weight_mask).transpose(-1, -2)

        query_layer1, key_layer1 = self.rotary_embeddings(query_layer1 * self.attention_head_size**-0.5, key_layer1)
        query_layer2, key_layer2 = self.rotary_embeddings(query_layer2 * self.attention_head_size**-0.5, key_layer2)

        attention_scores1 = torch.matmul(query_layer1, key_layer1.transpose(-1, -2))
        attention_scores2 = torch.matmul(query_layer2, key_layer2.transpose(-1, -2))
        del query_layer1, key_layer1, query_layer2, key_layer2
        torch.cuda.empty_cache()
        if attention_mask is not None:
            # Apply the attention mask is (precomputed for all layers in EsmModel forward() function)
            attention_scores1 = (
                attention_scores1 + attention_mask
            )  # if attention_mask has different shape: broadcasting
            attention_scores2 = (
                attention_scores2 + attention_mask
            )  # if attention_mask has different shape: broadcasting
            
        # Normalize the attention scores to probabilities.
        attention_probs = nn.functional.softmax(attention_scores1, dim=-1)-self.lambda_k*nn.functional.softmax(attention_scores2, dim=-1)
        del attention_scores1,attention_scores2
        torch.cuda.empty_cache()

        # Mask heads if we want to
        if head_mask is not None:
            attention_probs = attention_probs * head_mask
                
        context_layer = torch.matmul(
            attention_probs.to(value_layer.dtype), value_layer
        ) 
        weight_mask=(attention_probs>self.filter).to(dtype=context_layer.dtype)

        context_layer = context_layer.permute(0, 2, 1, 3).contiguous().view(context_layer.size()[:-2] + (self.all_head_size,))
        return ((context_layer, attention_probs, weight_mask))

hidden_size=1024
num_attention_heads = 16
attention_head_size = hidden_size // num_attention_heads
all_head_size = num_attention_heads * attention_head_size

query   = nn.Linear(hidden_size, all_head_size, bias=False)
key     = nn.Linear(hidden_size, all_head_size, bias=False)
value   = nn.Linear(hidden_size, all_head_size, bias=False)

rotary_embeddings = RotaryEmbedding(dim=attention_head_size)

#forward
hidden_states=torch.randn((5,512,1024))
attention_mask=None
head_mask = None
weight_mask=None
filter_delta=1/512



In [27]:
query_layer[0,0,:]

tensor([[-0.2611,  1.5992,  0.1804,  ...,  0.5968, -2.3944, -0.6930],
        [-0.1312,  1.8177, -0.3147,  ...,  0.1172, -1.8611, -0.2879],
        [ 0.5424,  1.3631, -0.4855,  ..., -0.1528, -1.8170,  0.3574],
        ...,
        [-0.1385,  0.3611,  0.3686,  ..., -0.6358, -0.7140, -0.2353],
        [-0.0990,  0.5525, -0.2358,  ...,  0.0520,  0.1448,  0.2591],
        [ 0.1324,  0.5278, -0.0677,  ...,  0.0496,  0.4965, -0.1348]],
       grad_fn=<SliceBackward0>)

In [22]:
torch.cuda.empty_cache()

In [21]:

query_layer = transpose_for_scores(query(hidden_states) ,num_attention_heads,attention_head_size)
key_layer   = transpose_for_scores(key(hidden_states)   ,num_attention_heads,attention_head_size)
value_layer = transpose_for_scores(value(hidden_states) ,num_attention_heads,attention_head_size)
if weight_mask is not None:
    query_layer =torch.matmul(query_layer.transpose(-1, -2),    weight_mask).transpose(-1, -2)
    key_layer   =torch.matmul(key_layer.transpose(-1, -2),      weight_mask).transpose(-1, -2)
    value_layer =torch.matmul(value_layer.transpose(-1, -2),    weight_mask).transpose(-1, -2)

# Matt: Our BERT model (which this code was derived from) scales attention logits down by sqrt(head_dim).
# ESM scales the query down by the same factor instead. Modulo numerical stability these are equivalent,
# but not when rotary embeddings get involved. Therefore, we scale the query here to match the original
# ESM code and fix rotary embeddings.
query_layer = query_layer * attention_head_size**-0.5
query_layer, key_layer = rotary_embeddings(query_layer, key_layer)

# Take the dot product between "query" and "key" to get the raw attention scores.
attention_scores = torch.matmul(query_layer, key_layer.transpose(-1, -2))

if attention_mask is not None:
    # Apply the attention mask is (precomputed for all layers in EsmModel forward() function)
    attention_scores = (
        attention_scores + attention_mask
    )  # if attention_mask has different shape: broadcasting

# Normalize the attention scores to probabilities.
attention_probs = nn.functional.softmax(attention_scores, dim=-1)

# Mask heads if we want to
if head_mask is not None:
    attention_probs = attention_probs * head_mask

context_layer = torch.matmul(
    attention_probs.to(value_layer.dtype), value_layer
) 

context_layer = context_layer.permute(0, 2, 1, 3).contiguous()
new_context_layer_shape = context_layer.size()[:-2] + (all_head_size,)
context_layer = context_layer.view(new_context_layer_shape)
weight_mask=(attention_probs>filter_delta).to(dtype=hidden_states.dtype)
outputs = ((context_layer, attention_probs,weight_mask))


In [20]:
outputs

(tensor([[[ 1.0813e+00,  2.0803e+00, -1.2510e+00,  ..., -6.2017e-02,
            2.1626e-02,  1.0763e-01],
          [ 1.1441e+00,  2.0327e+00, -1.3977e+00,  ..., -6.2017e-02,
            2.1626e-02,  1.0763e-01],
          [ 1.5474e+00,  1.9200e+00, -2.1274e+00,  ...,  1.8536e+00,
           -8.8839e-02,  8.4138e-01],
          ...,
          [ 1.1949e+00,  2.0757e+00, -3.1008e+00,  ..., -6.2017e-02,
            2.1626e-02,  1.0763e-01],
          [ 1.0012e+00,  1.1909e+00, -3.1914e+00,  ..., -6.2017e-02,
            2.1626e-02,  1.0763e-01],
          [ 1.0888e+00,  9.7920e-01, -4.6206e+00,  ..., -6.2017e-02,
            2.1626e-02,  1.0763e-01]],
 
         [[-8.3035e-01,  1.3039e-02, -1.1987e-01,  ...,  4.6316e-01,
            4.1246e-02, -1.4661e-01],
          [-8.5108e-02, -1.9981e-02, -3.5526e-03,  ...,  3.8266e-01,
            5.9157e-02, -9.6928e-01],
          [-8.5108e-02, -1.9981e-02, -3.5526e-03,  ...,  1.4847e-01,
            2.6842e-02, -6.2245e-02],
          ...,
    

In [108]:
torch.sum(weight_mask,dim=-2).shape

torch.Size([5, 16, 512])

In [105]:
weight_mask*(1/torch.sum(weight_mask,dim=-2))

RuntimeError: The size of tensor a (512) must match the size of tensor b (16) at non-singleton dimension 2

tensor([[[[ 2.5950e+00, -8.4891e-01, -4.6781e+00,  ..., -1.9121e-01,
            1.3399e-01, -3.8070e-01],
          [ 1.5076e+00, -2.8794e+00, -2.3673e-01,  ..., -2.1108e+00,
           -1.9021e+00,  1.7397e+00],
          [-3.3532e+00, -4.0036e-01, -8.2683e-01,  ..., -1.2093e+00,
           -3.0815e+00, -2.7635e+00],
          ...,
          [ 1.5560e+00,  1.5254e+00,  1.3848e+00,  ...,  9.2876e-01,
           -7.3604e-02, -7.9244e-01],
          [ 2.9704e+00,  1.8012e-01,  5.1598e-01,  ..., -1.8279e+00,
            4.2544e+00, -1.5749e+00],
          [-1.2356e+00, -2.2476e+00, -9.9806e-01,  ...,  2.4111e+00,
           -7.0737e-01,  2.2629e+00]],

         [[ 5.4006e-01,  1.4489e-01,  2.0610e+00,  ..., -1.2819e+00,
           -1.1399e+00, -1.2265e+00],
          [ 8.1464e-01, -1.9408e+00,  1.2839e+00,  ..., -9.1440e-01,
           -2.6337e+00, -2.0644e+00],
          [ 5.3869e-01, -2.9066e-01,  2.3781e+00,  ...,  2.7192e+00,
            1.1000e+00, -1.3441e+00],
          ...,
     

In [87]:
query_layer.shape

torch.Size([5, 16, 512, 64])

In [64]:
conv=nn.Conv1d(16,1,kernel_size=512)

In [67]:
conv.

Parameter containing:
tensor([0.0027], requires_grad=True)

In [57]:
(attention_probs>1/512).to(dtype=int).shape

torch.Size([5, 16, 512, 512])

In [72]:
torch.matmul(hidden_states,(attention_probs>1/512).to(dtype=int))

RuntimeError: The size of tensor a (5) must match the size of tensor b (16) at non-singleton dimension 1

In [11]:
import torch
import torch.nn as nn
from typing import Optional, Tuple

class MaskedLinear(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(MaskedLinear, self).__init__()
        self.linear = nn.Linear(input_dim, output_dim)

    def forward(self, x, mask=None):
        # 通过线性层
        x = self.linear(x)
        if mask is not None:
            # 根据mask取平均
            for indices in mask:
                avg_value = x[:, indices].mean(dim=1, keepdim=True)
                x[:, indices] = avg_value
        return x


def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat((-x2, x1), dim=-1)


def apply_rotary_pos_emb(x, cos, sin):
    """
    The function applies rotary positional embedding to the input tensor using cosine and sine values.

    :param x: The input tensor x. It is a 4-dimensional tensor with shape (batch_size, num_channels,
    height, width)
    :param cos: A tensor representing the cosine values for the rotary positional embedding. It has
    shape (batch_size, num_heads, sequence_length, embedding_dim)
    :param sin: The `sin` parameter is a tensor representing the sine values for the rotary positional
    embedding. It has shape `(batch_size, num_heads, sequence_length, embedding_dim)`
    :return: the result of applying the rotary positional embedding to the input tensor x.
    """
    cos = cos[:, :, : x.shape[-2], :]
    sin = sin[:, :, : x.shape[-2], :]
    return (x * cos) + (rotate_half(x) * sin)


class RotaryEmbedding(torch.nn.Module):
    """
    Rotary position embeddings based on those in
    [RoFormer](https://huggingface.co/docs/transformers/model_doc/roformer). Query and keys are transformed by rotation
    matrices which depend on their relative positions.
    """

    def __init__(self, dim: int):
        super().__init__()
        # Generate and save the inverse frequency buffer (non trainable)
        inv_freq = 1.0 / (10000 ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer("inv_freq", inv_freq)

        self._seq_len_cached = None
        self._cos_cached = None
        self._sin_cached = None

    def _update_cos_sin_tables(self, x, seq_dimension=2):
        seq_len = x.shape[seq_dimension]

        # Reset the tables if the sequence length has changed,
        # or if we're on a new device (possibly due to tracing for instance)
        if seq_len != self._seq_len_cached or self._cos_cached.device != x.device:
            self._seq_len_cached = seq_len
            t = torch.arange(x.shape[seq_dimension], device=x.device).type_as(
                self.inv_freq
            )
            freqs = torch.outer(t, self.inv_freq)
            emb = torch.cat((freqs, freqs), dim=-1).to(x.device)

            self._cos_cached = emb.cos()[None, None, :, :]
            self._sin_cached = emb.sin()[None, None, :, :]

        return self._cos_cached, self._sin_cached

    def forward(
        self, q: torch.Tensor, k: torch.Tensor
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        self._cos_cached, self._sin_cached = self._update_cos_sin_tables(
            k, seq_dimension=-2
        )

        return (
            apply_rotary_pos_emb(q, self._cos_cached, self._sin_cached),
            apply_rotary_pos_emb(k, self._cos_cached, self._sin_cached),
        )


class EsmSelfAttention(nn.Module):
    def __init__(self, config, position_embedding_type=None):
        super().__init__()
        if config.hidden_size % config.num_attention_heads != 0 and not hasattr(
            config, "embedding_size"
        ):
            raise ValueError(
                f"The hidden size ({config.hidden_size}) is not a multiple of the number of attention "
                f"heads ({config.num_attention_heads})"
            )

        self.num_attention_heads = config.num_attention_heads
        self.attention_head_size = int(config.hidden_size / config.num_attention_heads)
        self.all_head_size = self.num_attention_heads * self.attention_head_size

        self.query = MaskedLinear(config.hidden_size, self.all_head_size)
        self.key = MaskedLinear(config.hidden_size, self.all_head_size)
        self.value = MaskedLinear(config.hidden_size, self.all_head_size)

        self.dropout = nn.Dropout(config.attention_probs_dropout_prob)
        self.position_embedding_type = position_embedding_type or getattr(
            config, "position_embedding_type", "absolute"
        )
        self.rotary_embeddings = None
        if self.position_embedding_type in ["relative_key", "relative_key_query"]:
            self.max_position_embeddings = config.max_position_embeddings
            self.distance_embedding = nn.Embedding(
                2 * config.max_position_embeddings - 1, self.attention_head_size
            )
        elif self.position_embedding_type == "rotary":
            self.rotary_embeddings = RotaryEmbedding(dim=self.attention_head_size)

        self.is_decoder = config.is_decoder

    def transpose_for_scores(self, x: torch.Tensor) -> torch.Tensor:
        new_x_shape = x.size()[:-1] + (
            self.num_attention_heads,
            self.attention_head_size,
        )
        x = x.view(new_x_shape)
        return x.permute(0, 2, 1, 3)#[B,S,Hi]->[B,He,S,Hid]
    def filter_attention(self,attention_map):
        attention_map[attention_map>self.config.attention_limit]
        
    def forward(
        self,
        hidden_states: torch.Tensor,
        attention_mask: Optional[torch.FloatTensor] = None,
        head_mask: Optional[torch.FloatTensor] = None,
        encoder_hidden_states: Optional[torch.FloatTensor] = None,
        encoder_attention_mask: Optional[torch.FloatTensor] = None,
        past_key_value: Optional[Tuple[Tuple[torch.FloatTensor]]] = None,
        output_attentions: Optional[bool] = False,
        weight_mask=None

    ) -> Tuple[torch.Tensor]:
        mixed_query_layer = self.query(hidden_states)

        # If this is instantiated as a cross-attention module, the keys
        # and values come from an encoder; the attention mask needs to be
        # such that the encoder's padding tokens are not attended to.
        is_cross_attention = encoder_hidden_states is not None

        if is_cross_attention and past_key_value is not None:
            # reuse k,v, cross_attentions
            key_layer = past_key_value[0]
            value_layer = past_key_value[1]
            attention_mask = encoder_attention_mask
        elif is_cross_attention:
            key_layer = self.transpose_for_scores(self.key(encoder_hidden_states))
            value_layer = self.transpose_for_scores(self.value(encoder_hidden_states))
            attention_mask = encoder_attention_mask
        elif past_key_value is not None:
            key_layer = self.transpose_for_scores(self.key(hidden_states))
            value_layer = self.transpose_for_scores(self.value(hidden_states))
            key_layer = torch.cat([past_key_value[0], key_layer], dim=2)
            value_layer = torch.cat([past_key_value[1], value_layer], dim=2)
        else:
            key_layer = self.transpose_for_scores(self.key(hidden_states))
            value_layer = self.transpose_for_scores(self.value(hidden_states))

        query_layer = self.transpose_for_scores(mixed_query_layer)

        # Matt: Our BERT model (which this code was derived from) scales attention logits down by sqrt(head_dim).
        # ESM scales the query down by the same factor instead. Modulo numerical stability these are equivalent,
        # but not when rotary embeddings get involved. Therefore, we scale the query here to match the original
        # ESM code and fix rotary embeddings.
        query_layer = query_layer * self.attention_head_size**-0.5

        if self.position_embedding_type == "rotary":
            query_layer, key_layer = self.rotary_embeddings(query_layer, key_layer)

        # Take the dot product between "query" and "key" to get the raw attention scores.
        attention_scores = torch.matmul(query_layer, key_layer.transpose(-1, -2))

        if (
            self.position_embedding_type == "relative_key"
            or self.position_embedding_type == "relative_key_query"
        ):
            seq_length = hidden_states.size()[1]
            position_ids_l = torch.arange(
                seq_length, dtype=torch.long, device=hidden_states.device
            ).view(-1, 1)
            position_ids_r = torch.arange(
                seq_length, dtype=torch.long, device=hidden_states.device
            ).view(1, -1)
            distance = position_ids_l - position_ids_r
            positional_embedding = self.distance_embedding(
                distance + self.max_position_embeddings - 1
            )
            positional_embedding = positional_embedding.to(
                dtype=query_layer.dtype
            )  # fp16 compatibility

            if self.position_embedding_type == "relative_key":
                relative_position_scores = torch.einsum(
                    "bhld,lrd->bhlr", query_layer, positional_embedding
                )
                attention_scores = attention_scores + relative_position_scores
            elif self.position_embedding_type == "relative_key_query":
                relative_position_scores_query = torch.einsum(
                    "bhld,lrd->bhlr", query_layer, positional_embedding
                )
                relative_position_scores_key = torch.einsum(
                    "bhrd,lrd->bhlr", key_layer, positional_embedding
                )
                attention_scores = (
                    attention_scores
                    + relative_position_scores_query
                    + relative_position_scores_key
                )

        if attention_mask is not None:
            # Apply the attention mask is (precomputed for all layers in EsmModel forward() function)
            attention_scores = (
                attention_scores + attention_mask
            )  # if attention_mask has different shape: broadcasting

        # Normalize the attention scores to probabilities.
        attention_probs = nn.functional.softmax(attention_scores, dim=-1)

        # This is actually dropping out entire tokens to attend to, which might
        # seem a bit unusual, but is taken from the original Transformer paper.
        attention_probs = self.dropout(attention_probs)

        # Mask heads if we want to
        if head_mask is not None:
            attention_probs = attention_probs * head_mask

        context_layer = torch.matmul(
            attention_probs.to(value_layer.dtype), value_layer
        )  # TODO:make sure `.to(value_layer.dtype),`is okey.

        context_layer = context_layer.permute(0, 2, 1, 3).contiguous()
        new_context_layer_shape = context_layer.size()[:-2] + (self.all_head_size,)
        context_layer = context_layer.view(new_context_layer_shape)

        outputs = (
            (context_layer, attention_probs) if output_attentions else (context_layer,)
        )

        if self.is_decoder:
            outputs = outputs + (past_key_value,)
        return outputs

class EsmSelfOutput(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.dense = nn.Linear(config.hidden_size, config.hidden_size)
        self.dropout = nn.Dropout(config.hidden_dropout_prob)

    def forward(self, hidden_states, input_tensor):
        hidden_states = self.dense(hidden_states)
        hidden_states = self.dropout(hidden_states)
        hidden_states += input_tensor
        return hidden_states
    
class EsmAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.self = EsmSelfAttention(config)
        # self.self=EsmSelfAttentionAddFlashAttnPatch(config, position_embedding_type='rotary')
        self.output = EsmSelfOutput(config)
        self.pruned_heads = set()
        self.LayerNorm = nn.LayerNorm(config.hidden_size, eps=config.layer_norm_eps)

    def prune_heads(self, heads):
        if len(heads) == 0:
            return
        heads, index = find_pruneable_heads_and_indices(
            heads,
            self.self.num_attention_heads,
            self.self.attention_head_size,
            self.pruned_heads,
        )

        # Prune linear layers
        self.self.query = prune_linear_layer(self.self.query, index)
        self.self.key = prune_linear_layer(self.self.key, index)
        self.self.value = prune_linear_layer(self.self.value, index)
        self.output.dense = prune_linear_layer(self.output.dense, index, dim=1)

        # Update hyper params and store pruned heads
        self.self.num_attention_heads = self.self.num_attention_heads - len(heads)
        self.self.all_head_size = (
            self.self.attention_head_size * self.self.num_attention_heads
        )
        self.pruned_heads = self.pruned_heads.union(heads)

    def forward(
        self,
        hidden_states,
        attention_mask=None,
        head_mask=None,
        encoder_hidden_states=None,
        encoder_attention_mask=None,
        past_key_value=None,
        output_attentions=False,
        weight_mask=None
    ):
        hidden_states_ln = self.LayerNorm(hidden_states)
        self_outputs = self.self(
            hidden_states_ln,
            attention_mask,
            head_mask,
            encoder_hidden_states,
            encoder_attention_mask,
            past_key_value,
            output_attentions,
        )
        attention_output = self.output(self_outputs[0], hidden_states)
        outputs = (attention_output,) + self_outputs[
            1:
        ]  # add attentions if we output them
        return outputs


# 示例用法
input_dim = 5
output_dim = 5
model = MaskedLinear(input_dim, output_dim)

# 输入数据
X = torch.tensor([[1.0, 2.0, 3.0, 4.0, 5.0]])
# mask，表示需要取平均的位置
mask = [[0, 3], [2, 4]]

output = model(X, mask)
print(output)


tensor([[-0.2498, -3.1590, -0.3113, -0.2498, -0.3113]], grad_fn=<CopySlices>)
